In [8]:
df.nlargest(10, "Quantity")[[
    "InvoiceNo", "StockCode", "Quantity", "UnitPrice", "InvoiceDate"
]]

,InvoiceNo,StockCode,Quantity,UnitPrice,InvoiceDate
540421,581483,23843,80995,2.08,2011-12-09 09:15:00
61619,541431,23166,74215,1.04,2011-01-18 10:01:00
502122,578841,84826,12540,0.00,2011-11-25 15:57:00
74614,542504,37413,5568,0.00,2011-01-28 12:03:00
421632,573008,84077,4800,0.21,2011-10-27 12:26:00
206121,554868,22197,4300,0.72,2011-05-27 10:52:00
220843,556231,85123A,4000,0.00,2011-06-09 15:04:00
97432,544612,22053,3906,0.82,2011-02-22 10:43:00
270885,560599,18007,3186,0.06,2011-07-19 17:04:00
52711,540815,21108,3114,2.10,2011-01-11 12:55:00


In [10]:
# Preserve the original dataset for before/after comparison.
raw_df = df.copy()

# Print statistics before cleaning.
print("BEFORE CLEANING")
print(f"Rows: {len(raw_df):,}")
print(f"Columns: {raw_df.shape[1]:,}")
print(f"Duplicate rows: {raw_df.duplicated().sum():,}")
print(f"Rows with missing values: {raw_df.isna().any(axis=1).sum():,}")
print(f"Missing values by column:\n{raw_df.isna().sum()}")
print(f"Cancelled invoices: {raw_df['InvoiceNo'].astype(str).str.startswith('C').sum():,}")
print(f"Negative-quantity rows: {(raw_df['Quantity'] < 0).sum():,}")
print(f"Revenue before cleaning: {(raw_df['Quantity'] * raw_df['UnitPrice']).sum():,.2f}")

# Standardize fields required for transaction-level cleaning.
clean_df = raw_df.copy()
clean_df["InvoiceNo"] = clean_df["InvoiceNo"].astype(str).str.strip()
clean_df["StockCode"] = clean_df["StockCode"].astype(str).str.strip()
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"], errors="coerce")
clean_df["Quantity"] = pd.to_numeric(clean_df["Quantity"], errors="coerce")
clean_df["UnitPrice"] = pd.to_numeric(clean_df["UnitPrice"], errors="coerce")

# Remove exact duplicate records before classifying transactions.
clean_df = clean_df.drop_duplicates().copy()

# Identify cancelled and returned transactions before excluding them from sales.
clean_df["IsCancelled"] = clean_df["InvoiceNo"].str.startswith("C")
clean_df["IsReturned"] = clean_df["Quantity"] < 0
returns_df = clean_df[clean_df["IsCancelled"] | clean_df["IsReturned"]].copy()

# Remove rows missing any required field, including CustomerID.
required_columns = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
]
clean_df = clean_df.dropna(subset=required_columns).copy()

# Remove invalid prices, non-positive quantities, cancellations, and returns.
non_product_codes = {
    "POST", "DOT", "M", "MANUAL", "D", "DISCOUNT",
    "AMAZONFEE", "BANK CHARGES", "CRUK", "S"
}
clean_df["StockCodeUpper"] = clean_df["StockCode"].str.upper()
clean_df = clean_df[
    (~clean_df["IsCancelled"]) &
    (~clean_df["IsReturned"]) &
    (clean_df["Quantity"] > 0) &
    (clean_df["UnitPrice"] > 0) &
    (~clean_df["StockCodeUpper"].isin(non_product_codes))
].copy()

# Create revenue after invalid, cancelled, and returned transactions are excluded.
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["UnitPrice"]

# Print statistics after cleaning.
print("\nAFTER CLEANING")
print(f"Rows: {len(clean_df):,}")
print(f"Columns: {clean_df.shape[1]:,}")
print(f"Duplicate rows: {clean_df.duplicated().sum():,}")
print(f"Rows with missing values: {clean_df.isna().any(axis=1).sum():,}")
print(f"Missing values by column:\n{clean_df.isna().sum()}")
print(f"Cancelled/returned rows retained separately: {len(returns_df):,}")
print(f"Revenue after cleaning: {clean_df['Revenue'].sum():,.2f}")
print(f"Rows removed from sales dataset: {len(raw_df) - len(clean_df):,}")

# Display the first rows of the cleaned dataset.
display(clean_df.head())

BEFORE CLEANING
Rows: 541,909
Columns: 8
Duplicate rows: 5,268
Rows with missing values: 135,080
Missing values by column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
Cancelled invoices: 9,288
Negative-quantity rows: 10,624
Revenue before cleaning: 9,747,747.93

AFTER CLEANING
Rows: 391,286
Columns: 12
Duplicate rows: 0
Rows with missing values: 0
Missing values by column:
InvoiceNo         0
StockCode         0
Description       0
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
IsCancelled       0
IsReturned        0
StockCodeUpper    0
Revenue           0
dtype: int64
Cancelled/returned rows retained separately: 10,587
Revenue after cleaning: 8,743,913.64
Rows removed from sales dataset: 150,623


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled,IsReturned,StockCodeUpper,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,False,85123A,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,71053,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,False,84406B,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,84029G,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,84029E,20.34
